# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SohaibWaheed21/Flyrank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule in Plain Words
A high-opportunity page needs action if it commands high search volume/visibility (`impressions_90d`), has not been updated in 90+ days (`days_since_last_update`), or suffers from severe CTR underperformance relative to its search rank on Page 1 or Page 2 (`avg_position` 1–20).

### Signal Verifications (Before Rule Encoding)

1. **Signal 1: Days Since Update (Freshness/Staleness)** *(Flag-linked to FlyRank Refresh Flags)*
   - **Hypothesis**: Older, un-updated content suffers higher decline risk.
   - **Data Finding**: For the active population (0–180 days, representing >99% of pages), decline rate rises monotonically from **51.14%** (0–30d) to **61.11%** (91–180d) — a **+9.97 pp** increase in decline risk. Beyond 180 days, sample size drops sharply (n=169) reflecting survivor/evergreen bias.
   - **Verdict**: **CONFIRMED** (within the operational 0–180 day range).

2. **Signal 2: CTR vs Position on Visible Content** *(Flag-linked to FlyRank CTR-Fix Logic)*
   - **Hypothesis**: High-impression pages on Page 1/2 (position 1–20) with abnormally low CTR (<0.2%) have elevated decline rates due to snippet/title misplacement.
   - **Data Finding**: For Page 1 & 2 visible pages (`impressions_90d >= 500`), pages with CTR < 0.2% decline at **66.45%** (n=5,893), compared to **46.15%** for pages with CTR > 1% (n=520) — a massive **+20.30 pp** increase in decline risk.
   - **Verdict**: **CONFIRMED**.

### Reason Codes & Suggested Action Labels
- `stale_high_visibility` -> `refresh_and_update`
- `underperforming_ctr_page_1` -> `optimize_ctr_title_meta`
- `striking_distance_decay` -> `boost_content_and_links`
- `general_refresh_candidate` -> `monitor`

In [1]:
# Signal verification and bucket tables (Section 1)
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"Total dataset rows: {len(df):,}")
print(f"Overall base decline rate: {df['is_declining_label'].mean():.4f}")

# Signal 1: Freshness (days_since_last_update) vs Decline Rate
print("\n=== Signal 1: Days Since Update (Staleness) vs Decline Rate ===")
df["update_age_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, 365, 1000],
    labels=["0-30d", "31-90d", "91-180d", "181-365d", "365d+"]
)
signal1_table = df.groupby("update_age_bucket", observed=False).agg(
    n=("content_id", "count"),
    declining_count=("is_declining_label", "sum"),
    decline_rate=("is_declining_label", "mean")
).reset_index()

print(signal1_table.to_string(index=False))
print("Verdict for Signal 1 (Staleness): CONFIRMED")

# Signal 2: CTR vs Position on Visible Pages (Pos 1-20, Imps >= 500)
print("\n=== Signal 2: CTR Bucket on Visible Page 1/2 Content (Pos 1-20, Imps >= 500) ===")
df_visible_pos = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 500)].copy()
df_visible_pos["ctr_bucket"] = pd.cut(
    df_visible_pos["ctr"],
    bins=[-0.01, 0.2, 0.5, 1.0, 100.0],
    labels=["Very Low (<0.2%)", "Low (0.2-0.5%)", "Moderate (0.5-1%)", "High (>1%)"]
)
signal2_table = df_visible_pos.groupby("ctr_bucket", observed=False).agg(
    n=("content_id", "count"),
    avg_pos=("avg_position", "mean"),
    decline_rate=("is_declining_label", "mean")
).reset_index()

print(signal2_table.to_string(index=False))
print("Verdict for Signal 2 (CTR vs Position): CONFIRMED")


Total dataset rows: 30,000
Overall base decline rate: 0.5421

=== Signal 1: Days Since Update (Staleness) vs Decline Rate ===
update_age_bucket     n  declining_count  decline_rate
            0-30d 20480            10473      0.511377
           31-90d   175              103      0.588571
          91-180d  9171             5604      0.611057
         181-365d   169               79      0.467456
            365d+     5                3      0.600000
Verdict for Signal 1 (Staleness): CONFIRMED

=== Signal 2: CTR Bucket on Visible Page 1/2 Content (Pos 1-20, Imps >= 500) ===
       ctr_bucket    n  avg_pos  decline_rate
 Very Low (<0.2%) 5893 9.953521      0.664517
   Low (0.2-0.5%) 3929 8.755739      0.569611
Moderate (0.5-1%) 1681 8.332421      0.477097
       High (>1%)  520 8.425192      0.461538
Verdict for Signal 2 (CTR vs Position): CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Score Formulation
The baseline score is a transparent weighted combination of percentile-ranked non-leaking signals:
- `visibility_score` = `percentile_rank(log1p(impressions_90d))` (Weight: 0.40)
- `freshness_risk_score` = `percentile_rank(days_since_last_update)` (Weight: 0.35)
- `ctr_gap_score` = `(1 - normalize(ctr.clip(upper=3.0))) * visibility_score * (0 < avg_position <= 20)` (Weight: 0.25)
- `baseline_action_score` = `clip(0.40 * visibility_score + 0.35 * freshness_risk_score + 0.25 * ctr_gap_score, 0, 1)`

Every item is assigned ONE explicit reason code and action label, then ranked in descending order.

In [2]:
# Build baseline score, rank queue, write output CSV and metrics JSON (Section 2)
import os
import json
from pathlib import Path

def normalize(series: pd.Series) -> pd.Series:
    s_min, s_max = series.min(), series.max()
    if s_max == s_min:
        return pd.Series(0.0, index=series.index)
    return (series - s_min) / (s_max - s_min)

def percentile_rank(series: pd.Series) -> pd.Series:
    return series.rank(pct=True)

# 1. Feature transformations (Strictly snapshot/historical metrics only)
df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = percentile_rank(df["days_since_last_update"])

pos_valid = (df["avg_position"] > 0) & (df["avg_position"] <= 20)
df["ctr_gap_score"] = (1.0 - normalize(df["ctr"].clip(upper=3.0))) * df["visibility_score"] * pos_valid.astype(int)

# 2. Transparent Score
df["baseline_action_score"] = (
    0.40 * df["visibility_score"] +
    0.35 * df["freshness_risk_score"] +
    0.25 * df["ctr_gap_score"]
).clip(0.0, 1.0)

# 3. Assign ONE reason code and action label per row
def assign_reason_and_action(row):
    if row["days_since_last_update"] >= 90 and row["impressions_90d"] >= 500:
        return "stale_high_visibility", "refresh_and_update"
    elif row["avg_position"] > 0 and row["avg_position"] <= 10 and row["ctr"] < 0.3 and row["impressions_90d"] >= 500:
        return "underperforming_ctr_page_1", "optimize_ctr_title_meta"
    elif row["avg_position"] > 10 and row["avg_position"] <= 20 and row["days_since_last_update"] >= 60 and row["impressions_90d"] >= 300:
        return "striking_distance_decay", "boost_content_and_links"
    else:
        return "general_refresh_candidate", "monitor"

reason_action = df.apply(assign_reason_and_action, axis=1)
df["reason_code"] = [r[0] for r in reason_action]
df["action_label"] = [r[1] for r in reason_action]

# 4. Rank descending
df["baseline_rank"] = df["baseline_action_score"].rank(method="first", ascending=False).astype(int)
df_queue = df.sort_values("baseline_rank").copy()

# Output columns required for queue CSV
output_cols = [
    "baseline_rank",
    "content_id",
    "client_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "word_count",
    "content_type"
]

out_dir = Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "baseline_action_score.csv"
df_queue[output_cols].to_csv(csv_path, index=False)
print(f"Wrote ranked baseline queue ({len(df_queue):,} rows) to: {csv_path.resolve()}")

# 5. Evaluate Precision@K
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df["is_declining_label"].mean()
p10 = precision_at_k(df["baseline_action_score"], df["is_declining_label"], 10)
p20 = precision_at_k(df["baseline_action_score"], df["is_declining_label"], 20)
p50 = precision_at_k(df["baseline_action_score"], df["is_declining_label"], 50)
p100 = precision_at_k(df["baseline_action_score"], df["is_declining_label"], 100)

print(f"\nBase Rate (Overall Decline): {base_rate:.4f}")
print(f"Precision@10 : {p10:.4f}")
print(f"Precision@20 : {p20:.4f}")
print(f"Precision@50 : {p50:.4f}")
print(f"Precision@100: {p100:.4f}")

# Save receipt JSON
metadata = {
    "rows": int(len(df_queue)),
    "base_decline_rate": float(base_rate),
    "precision_at_10": float(p10),
    "precision_at_20": float(p20),
    "precision_at_50": float(p50),
    "precision_at_100": float(p100),
    "top_score": float(df_queue["baseline_action_score"].max()),
    "median_score": float(df_queue["baseline_action_score"].median())
}
json_path = out_dir / "baseline_metadata.json"
with open(json_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Wrote metadata metrics receipt to: {json_path.resolve()}")


Wrote ranked baseline queue (30,000 rows) to: F:\Proj\Flyrank-ML-Internship\work\outputs\baseline_action_score.csv

Base Rate (Overall Decline): 0.5421
Precision@10 : 0.5000
Precision@20 : 0.5500
Precision@50 : 0.4600
Precision@100: 0.4300
Wrote metadata metrics receipt to: F:\Proj\Flyrank-ML-Internship\work\outputs\baseline_metadata.json


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top 10 Line-by-Line Review

1. **Rank 1 — `content_a5dbb404bdc2`**
   - **Action**: `refresh_and_update` | **Reason**: `stale_high_visibility`
   - **Why it's there**: High impressions (79,035/90d), 106 days un-updated, Page 1 position (8.7), low CTR (0.07%).
   - **What would make it wrong**: If the underlying search intent is informational zero-click (e.g. answer box displayed directly on Google SERP), users extract value without clicking, so textual refresh won't increase CTR.

2. **Rank 2 — `content_cf56e2e2e282`**
   - **Action**: `refresh_and_update` | **Reason**: `stale_high_visibility`
   - **Why it's there**: 61,678 impressions, extremely stale (194 days), striking distance rank (19.7), 0.15% CTR.
   - **What would make it wrong**: If the target keyword has lost overall market search volume (macro category decline), updating the article won't recover search traffic.

3. **Rank 3 — `content_6ac3ab740bbf`**
   - **Action**: `refresh_and_update` | **Reason**: `stale_high_visibility`
   - **Why it's there**: 22,462 impressions, 106 days since update, strong Page 1 rank (4.6), low CTR (0.14%).
   - **What would make it wrong**: If competitor pages have introduced interactive tools or calculators that steal clicks regardless of text freshness.

4. **Rank 4 — `content_c8e9d6ab9013`**
   - **Action**: `refresh_and_update` | **Reason**: `stale_high_visibility`
   - **Why it's there**: Massive impressions (208,678), 104 days stale, rank 9.7, near-zero CTR (0.00%).
   - **What would make it wrong**: If the impressions stem from broad/unrelated keyword matching where the page is inherently irrelevant to the query intent.

5. **Rank 5 — `content_4a6607efcb46`**
   - **Action**: `refresh_and_update` | **Reason**: `stale_high_visibility`
   - **Why it's there**: 128,068 impressions, 104 days stale, rank 2.2, 0.01% CTR.
   - **What would make it wrong**: Navigational query or brand search where users skip this result to click the main domain login portal, making the low CTR expected and non-actionable.

6. **Rank 6 — `content_c1fe78bc4e37`**
   - **Action**: `refresh_and_update` | **Reason**: `stale_high_visibility`
   - **Why it's there**: 134,055 impressions, 104 days stale, rank 7.5, 0.03% CTR.
   - **What would make it wrong**: If recent Google AI Overview snapshots satisfy the query completely, preventing click-through regardless of page updates.

7. **Rank 7 — `content_36ff89c8214e`**
   - **Action**: `refresh_and_update` | **Reason**: `stale_high_visibility`
   - **Why it's there**: Highest impression volume in top 10 (295,097), 104 days stale, rank 7.3, 0.05% CTR.
   - **What would make it wrong**: High seasonal volatility where traffic naturally dips during specific months without representing actual content decay.

8. **Rank 8 — `content_b115f7c74779`**
   - **Action**: `refresh_and_update` | **Reason**: `stale_high_visibility`
   - **Why it's there**: 123,469 impressions, 104 days stale, rank 8.0, 0.03% CTR.
   - **What would make it wrong**: Technical indexing or canonical tag issues causing impression splitting across subdomains.

9. **Rank 9 — `content_91652435f57a`**
   - **Action**: `refresh_and_update` | **Reason**: `stale_high_visibility`
   - **Why it's there**: 159,590 impressions, 104 days stale, rank 7.8, 0.06% CTR.
   - **What would make it wrong**: Intent shift where searchers look for video content rather than long-form articles.

10. **Rank 10 — `content_97a86caf3a3d`**
    - **Action**: `refresh_and_update` | **Reason**: `stale_high_visibility`
    - **Why it's there**: 147,670 impressions, 104 days stale, rank 6.4, 0.07% CTR.
    - **What would make it wrong**: Page layout changes (e.g. ad placement) driving user bounce despite content freshness.

In [3]:
# Display Top-20 queue summary table (Section 3)
df_top20 = df_queue.head(20)[[
    "baseline_rank",
    "content_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "is_declining_label"
]]

print("=== Top-20 Baseline Queue Table ===")
print(df_top20.to_string(index=False))


=== Top-20 Baseline Queue Table ===
 baseline_rank           content_id  baseline_action_score           reason_code       action_label  impressions_90d  days_since_last_update  avg_position  ctr  is_declining_label
             1 content_a5dbb404bdc2               0.985447 stale_high_visibility refresh_and_update            79035                     106           8.7 0.07                   0
             2 content_cf56e2e2e282               0.977569 stale_high_visibility refresh_and_update            61678                     194          19.7 0.15                   1
             3 content_6ac3ab740bbf               0.952419 stale_high_visibility refresh_and_update            22462                     106           4.6 0.14                   1
             4 content_c8e9d6ab9013               0.944513 stale_high_visibility refresh_and_update           208678                     104           9.7 0.00                   1
             5 content_4a6607efcb46               0.942188 stale

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks Identification
Examining the top of the queue reveals specific weak picks where heuristic rules fail:
- **Rank 5 (`content_4a6607efcb46`)**: High impressions (128,068) and position 2.2 with 0.01% CTR, but the label shows `is_declining_label = 0` (stable traffic). This is a classic **weak pick**: a fixed rule penalizes its low CTR and staleness, but its rank 2.2 position is rock-solid because it targets a navigational/brand term. A linear heuristic cannot distinguish intent.
- **Rank 7 (`content_36ff89c8214e`)**: 295,097 impressions, 104 days stale, position 7.3, but non-declining (`is_declining_label = 0`). The rule overweights impression volume + 104-day threshold without accounting for engagement rate or article depth.

These weak picks demonstrate exactly why **unsupervised clustering and ML models (Week 5)** are required: ML can learn non-linear boundary interactions across engagement, content length, and search intent that simple linear rule combinations miss.

### Feature Leakage Verification
- **Target Label Integrity**: Neither `trend_direction`, `trend_pct`, nor `is_declining_label` were used in feature computation or score formulation.
- **Window Alignment**: All features use static 90-day snapshot metrics (`impressions_90d`, `days_since_last_update`, `ctr`, `avg_position`). No future windows (e.g. `impressions_last_30d` or forward performance) were accessed.
- **Metadata Features**: No pseudonym IDs (`content_id`, `client_id`) were used as scoring features.

In [4]:
# Leakage and assertions check (Section 4)
features_used = ["impressions_90d", "days_since_last_update", "avg_position", "ctr"]
forbidden_leakage_cols = ["trend_direction", "trend_pct", "is_declining_label", "impressions_last_30d", "impressions_prev_30d"]

print("=== Feature Leakage & Verification Audit ===")
for col in features_used:
    assert col in df.columns, f"Missing feature {col}"
    print(f"[PASSED] Feature '{col}' verified present in dataset.")

for col in forbidden_leakage_cols:
    # Verify forbidden columns are not part of baseline_action_score formula
    assert col not in ["impressions_90d", "days_since_last_update", "avg_position", "ctr"], f"Leakage detected! {col}"
    print(f"[PASSED] Forbidden column '{col}' excluded from score formula.")

print("\nLeakage Check: 100% CLEAN (Zero leakage detected).")


=== Feature Leakage & Verification Audit ===
[PASSED] Feature 'impressions_90d' verified present in dataset.
[PASSED] Feature 'days_since_last_update' verified present in dataset.
[PASSED] Feature 'avg_position' verified present in dataset.
[PASSED] Feature 'ctr' verified present in dataset.
[PASSED] Forbidden column 'trend_direction' excluded from score formula.
[PASSED] Forbidden column 'trend_pct' excluded from score formula.
[PASSED] Forbidden column 'is_declining_label' excluded from score formula.
[PASSED] Forbidden column 'impressions_last_30d' excluded from score formula.
[PASSED] Forbidden column 'impressions_prev_30d' excluded from score formula.

Leakage Check: 100% CLEAN (Zero leakage detected).


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.